In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "01. res_analysis_loo"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from joblib import load, dump
import os
import tables
import pathlib
from IPython.display import display
from matplotlib.colors import Normalize
from matplotlib import cm
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde
from sklearn.metrics import r2_score


import warnings
warnings.filterwarnings("ignore")
    
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr

In [ ]:
# Version check cell for the key libraries we use
import platform, importlib

libs = [
    ("Python", None),          # handled separately
    ("numpy", "numpy"),
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("seaborn", "seaborn"),
    ("scipy", "scipy"),
    ("scikit-learn", "sklearn"),
    ("joblib", "joblib"),
    ("PyTables (tables)", "tables"),
    ("numexpr", "numexpr"),
    ("IPython", "IPython"),
]

def get_version(modname: str) -> str:
    try:
        m = importlib.import_module(modname)
    except Exception:
        return "not installed"
    for attr in ("__version__", "version", "VERSION"):
        v = getattr(m, attr, None)
        if isinstance(v, str):
            return v
        if v is not None:
            try:
                return str(v)
            except Exception:
                pass
    # fallback to importlib.metadata
    try:
        from importlib.metadata import version as md_version
        return md_version(modname)
    except Exception:
        return "unknown"

rows = []
for display_name, import_name in libs:
    if display_name == "Python":
        rows.append((display_name, platform.python_version()))
    else:
        rows.append((display_name, get_version(import_name)))
print(pd.DataFrame(rows, columns=["Library", "Version"]).to_string(index=False))


### Datasets

In [ ]:
corrs = load('../data/correlations.joblib')

In [ ]:
proteomics = load('../data/proteomics_matched.joblib')

In [ ]:
tables.file._open_files.close_all()
p = pathlib.Path(r"../data/enet_all_results.h5")
print("exists:", p.exists(), "size:", p.stat().st_size, "bytes")
# Load all data from the HDF5 file
h5_file = '../data/enet_all_results.h5'
eps = 1e-8

with pd.HDFStore(h5_file) as store:
    keys = store.keys()
    df_list = [store[key] for key in keys]
    combined_df = pd.concat(df_list, ignore_index=True)
print("Combined DataFrame shape:", combined_df.shape)


In [ ]:
df_list

In [ ]:
combined_df

In [ ]:
# combined df name (change if yours is different)
df = combined_df

# Compute per-protein MSE over all its samples
mse_by_protein = (
    df.assign(sq_err=(df["true"].astype(float) - df["predicted"].astype(float)) ** 2)
      .groupby("protein")["sq_err"]
      .mean()
      .sort_values()
)

top100 = mse_by_protein.head(100).index.astype(str).tolist()

# Write protein_list.txt (one protein per line)
with open("protein_list.txt", "w") as f:
    f.write("\n".join(top100) + "\n")

print("Wrote protein_list.txt with", len(top100), "proteins.")


In [ ]:
# --- Parameter usage summary by protein ---
import json

# We copy so the original isn't touched
df = combined_df.copy()

# We parse params_dict (it may be a dict already or a JSON-like string)
def _to_params(d):
    if isinstance(d, dict):
        return d
    if isinstance(d, str):
        # handle either valid JSON or python-ish dict strings
        try:
            return json.loads(d)
        except json.JSONDecodeError:
            try:
                return json.loads(d.replace("'", '"'))
            except Exception:
                return {}
    return {}

df["_params"] = df["params_dict"].apply(_to_params)
df["alpha"] = df["_params"].apply(lambda p: p.get("alpha"))
df["l1_ratio"] = df["_params"].apply(lambda p: p.get("l1_ratio"))

# We compute counts per protein for alpha values
alpha_counts = (
    df.groupby(["protein", "alpha"])
      .size()
      .reset_index(name="count")
      .pivot(index="protein", columns="alpha", values="count")
      .fillna(0)
      .astype(int)
      .sort_index()
)
alpha_counts.columns = [f"alpha={c}" for c in alpha_counts.columns]

# We compute counts per protein for l1_ratio values
l1_counts = (
    df.groupby(["protein", "l1_ratio"])
      .size()
      .reset_index(name="count")
      .pivot(index="protein", columns="l1_ratio", values="count")
      .fillna(0)
      .astype(int)
      .sort_index()
)
l1_counts.columns = [f"l1_ratio={c}" for c in l1_counts.columns]

# (Optional) We compute counts for each (alpha, l1_ratio) combination per protein
combo_counts = (
    df.groupby(["protein", "alpha", "l1_ratio"])
      .size()
      .reset_index(name="count")
      .sort_values(["protein", "count"], ascending=[True, False])
)

# We display the three summary tables
display(alpha_counts)
display(l1_counts)
display(combo_counts)


## Rescaled Analysis, 

scTranslator-style per-sample min–max normalization.

they achieve the following results with 10k model: 20000 RNA -> 1000 Protein
Test Set: AVG mse 0.0487, AVG ccc 0.8979

#### Rescale [10e-8, 1]

In [ ]:
# for scaling using truth min/max scale
# EPS = 1e-8

# df = combined_df.copy()
# df["true"] = pd.to_numeric(df["true"], errors="coerce")
# df["predicted"] = pd.to_numeric(df["predicted"], errors="coerce")

# # basic normalization with supplied min/max
# def normalization_with_minmax(x: pd.Series, xmin: float, xmax: float,
#                               low: float = EPS, high: float = 1.0) -> pd.Series:
#     if pd.isna(xmin) or pd.isna(xmax) or xmax == xmin:
#         return pd.Series(low, index=x.index, dtype="float64")
#     return low + (x - xmin) / (xmax - xmin) * (high - low)

# mm = df.groupby("LOO sample", observed=True)["true"].agg(tmin="min", tmax="max")
# df = df.merge(mm, left_on="LOO sample", right_index=True, how="left")

# # true uses its own min/max; pred uses truth's min/max
# df["true_mm"] = df.groupby("LOO sample", observed=True).apply(
#     lambda g: normalization_with_minmax(g["true"], g["tmin"].iloc[0], g["tmax"].iloc[0])
# ).reset_index(level=0, drop=True)

# df["pred_mm"] = df.groupby("LOO sample", observed=True).apply(
#     lambda g: normalization_with_minmax(g["predicted"], g["tmin"].iloc[0], g["tmax"].iloc[0])
# ).reset_index(level=0, drop=True)

# df["true_mm"] = df["true_mm"].clip(lower=EPS, upper=1.0)
# df["pred_mm"] = df["pred_mm"].clip(lower=EPS, upper=1.0)


In [ ]:
# to scale independently per sample using min/max of each vector, true and predicted separately
# scTranslator-style per-sample min–max normalization for a long DataFrame
# expected columns in `combined_df`: ["protein", "LOO sample", "true", "predicted"]

EPS = 1e-8  # lower bound in the target scale

# --- sanity checks
required_cols = {"protein", "LOO sample", "true", "predicted"}
missing = required_cols - set(combined_df.columns)
if missing:
    raise ValueError(f"combined_df is missing required columns: {sorted(missing)}")

df = combined_df.copy()

# ensure numeric
df["true"] = pd.to_numeric(df["true"], errors="coerce")
df["predicted"] = pd.to_numeric(df["predicted"], errors="coerce")

# scTranslator-style normalization, safe for constant vectors / NaNs
def normalization(x: pd.Series, low: float = EPS, high: float = 1.0) -> pd.Series:
    # x is a Series for a single sample's vector (across proteins)
    xmin = x.min(skipna=True)
    xmax = x.max(skipna=True)
    if pd.isna(xmin) or pd.isna(xmax):
        # all NaNs -> return NaNs (or low, if you prefer)
        return pd.Series(np.nan, index=x.index, dtype="float64")
    if xmax == xmin:
        # constant vector -> map everything to `low`
        return pd.Series(low, index=x.index, dtype="float64")
    return low + (x - xmin) / (xmax - xmin) * (1.0 - low)

# normalize per sample, separately for true and predicted
df["true_mm"] = df.groupby("LOO sample", observed=True)["true"].transform(normalization)
df["pred_mm"] = df.groupby("LOO sample", observed=True)["predicted"].transform(normalization)

# clip to be extra-safe against tiny floating drift
df["true_mm"] = df["true_mm"].clip(lower=EPS, upper=1.0)
df["pred_mm"] = df["pred_mm"].clip(lower=EPS, upper=1.0)

# quick validation
print("Ranges after scaling:")
print("  true_mm :", float(np.nanmin(df["true_mm"])), "→", float(np.nanmax(df["true_mm"])))
print("  pred_mm :", float(np.nanmin(df["pred_mm"])), "→", float(np.nanmax(df["pred_mm"])))

# OPTIONAL: per-protein Pearson r on the min–max scale
want_metrics = True
if want_metrics:
    t_mat = df.pivot(index="LOO sample", columns="protein", values="true_mm")
    p_mat = df.pivot(index="LOO sample", columns="protein", values="pred_mm")

    common = t_mat.columns.intersection(p_mat.columns)
    t_mat = t_mat[common]
    p_mat = p_mat[common]

    t = t_mat.to_numpy(dtype=float)
    p = p_mat.to_numpy(dtype=float)

    t_z = t - np.nanmean(t, axis=0, keepdims=True)
    p_z = p - np.nanmean(p, axis=0, keepdims=True)

    num = np.nansum(t_z * p_z, axis=0)
    den = np.sqrt(np.nansum(t_z**2, axis=0) * np.nansum(p_z**2, axis=0))
    r = np.divide(num, den, out=np.full_like(num, np.nan, dtype=float), where=den!=0)

    per_protein_corr = pd.Series(r, index=common, name="pearson_r").sort_values(ascending=False)
    print("\nPer-protein Pearson r (first 10):")
    print(per_protein_corr.head(10))

# df now contains:
#   - 'true_mm' : truth scaled to [1e-8, 1] per sample using its *own* min/max
#   - 'pred_mm' : predictions scaled to [1e-8, 1] per sample using their *own* min/max


In [ ]:
# === Per-protein Pearson r AND Cosine similarity (NaN-safe), plus MSE ===
# Uses the min–max columns if present; otherwise falls back to raw.

TRUE_COL = "true_mm"  if "true_mm"  in df.columns else "true"
PRED_COL = "pred_mm"  if "pred_mm"  in df.columns else "predicted"
print(f"Evaluating on columns: TRUE={TRUE_COL}  PRED={PRED_COL}")

# Pivot to samples × proteins matrices
T = df.pivot(index="LOO sample", columns="protein", values=TRUE_COL).astype(float)
P = df.pivot(index="LOO sample", columns="protein", values=PRED_COL).astype(float)

# Align columns
common = T.columns.intersection(P.columns)
T = T[common]
P = P[common]

t = T.to_numpy(dtype=float)
p = P.to_numpy(dtype=float)

# ---------- Pearson r (column-wise) ----------
t_z = t - np.nanmean(t, axis=0, keepdims=True)
p_z = p - np.nanmean(p, axis=0, keepdims=True)

num_r = np.nansum(t_z * p_z, axis=0)
den_r = np.sqrt(np.nansum(t_z**2, axis=0) * np.nansum(p_z**2, axis=0))
pearson_r = np.divide(num_r, den_r, out=np.full_like(num_r, np.nan, dtype=float), where=den_r != 0)

# ---------- Cosine similarity (column-wise; uncentered) ----------
mask = ~(np.isnan(t) | np.isnan(p))
num_c = np.nansum(np.where(mask, t * p, 0.0), axis=0)
t_norm = np.sqrt(np.nansum(np.where(mask, t * t, 0.0), axis=0))
p_norm = np.sqrt(np.nansum(np.where(mask, p * p, 0.0), axis=0))
den_c = t_norm * p_norm
cosine = np.divide(num_c, den_c, out=np.full_like(num_c, np.nan, dtype=float), where=den_c != 0)

# ---------- MSE (column-wise) ----------
mse = np.nanmean((t - p) ** 2, axis=0)

# Assemble table
per_protein_metrics = pd.DataFrame(
    {"pearson_r": pearson_r, "cosine": cosine, "mse": mse},
    index=common
).sort_values("cosine", ascending=False)

# Show top by cosine
TOP_N = 30
print(f"\nTop {TOP_N} proteins by cosine similarity:")
display(per_protein_metrics.head(TOP_N))

# Also useful: top by Pearson (for comparison)
print(f"\nTop {TOP_N} proteins by Pearson r:")
display(per_protein_metrics.sort_values("pearson_r", ascending=False).head(TOP_N))

# Save full table
per_protein_metrics.to_csv("per_protein_metrics_pearson_cosine_mse.csv")
print("\nSaved: per_protein_metrics_pearson_cosine_mse.csv")


In [ ]:
# Add R^2 from Pearson r and save an updated CSV

assert "per_protein_metrics" in globals(), "per_protein_metrics not found. Run the metrics cell first."
m = per_protein_metrics.copy()

m["r2"] = np.square(m["pearson_r"])
per_protein_metrics = m  # keep the name consistent downstream

per_protein_metrics.to_csv("per_protein_metrics_pearson_cosine_mse_r2.csv")
print("Saved: per_protein_metrics_pearson_cosine_mse_r2.csv")
display(per_protein_metrics.head())


In [ ]:
# Histogram of per-protein Pearson r

vals = per_protein_metrics["pearson_r"].dropna().to_numpy()
mean_v = float(np.nanmean(vals)) if len(vals) else np.nan
med_v  = float(np.nanmedian(vals)) if len(vals) else np.nan

plt.figure()
plt.hist(vals, bins=30)
if np.isfinite(mean_v):
    plt.axvline(mean_v, linestyle="--", linewidth=1.5, label=f"mean = {mean_v:.3f}")
if np.isfinite(med_v):
    plt.axvline(med_v, linestyle="-.", linewidth=1.5, label=f"median = {med_v:.3f}")
plt.xlabel("Pearson r")
plt.ylabel("Count")
plt.title("Per-protein Pearson r distribution")
plt.legend()
plt.tight_layout()
plt.show()

print(f"n={len(vals)} | mean={mean_v:.3f} | median={med_v:.3f} | std={np.nanstd(vals):.3f}")


In [ ]:
# === Plot cosine similarity distribution ===
vals = per_protein_metrics["cosine"].dropna().to_numpy()
mean_v = float(np.nanmean(vals)) if len(vals) else np.nan
med_v  = float(np.nanmedian(vals)) if len(vals) else np.nan

plt.figure()
plt.hist(vals, bins=30)
if np.isfinite(mean_v):
    plt.axvline(mean_v, linestyle="--", linewidth=1.5, label=f"mean = {mean_v:.3f}")
if np.isfinite(med_v):
    plt.axvline(med_v, linestyle="-.", linewidth=1.5, label=f"median = {med_v:.3f}")
plt.xlabel("Cosine similarity")
plt.ylabel("Count")
plt.title("Per-protein cosine similarity distribution")
plt.legend()
plt.tight_layout()
plt.show()

print(f"n={len(vals)} | mean={mean_v:.3f} | median={med_v:.3f} | std={np.nanstd(vals):.3f}")


In [ ]:
# === Plot mse distribution ===
vals = per_protein_metrics["mse"].dropna().to_numpy()
mean_v = float(np.nanmean(vals)) if len(vals) else np.nan
med_v  = float(np.nanmedian(vals)) if len(vals) else np.nan

plt.figure()
plt.hist(vals, bins=30)
if np.isfinite(mean_v):
    plt.axvline(mean_v, linestyle="--", linewidth=1.5, label=f"mean = {mean_v:.6g}")
if np.isfinite(med_v):
    plt.axvline(med_v, linestyle="-.", linewidth=1.5, label=f"median = {med_v:.6g}")
plt.xlabel("MSE")
plt.ylabel("Count")
plt.title("Per-protein MSE distribution")
plt.legend()
plt.tight_layout()
plt.show()

print(f"n={len(vals)} | mean={mean_v:.6g} | median={med_v:.6g} | std={np.nanstd(vals):.6g}")


In [ ]:
# Summary counts/percentages for cosine, R^2, and MSE thresholds + metric means

assert "per_protein_metrics" in globals(), "per_protein_metrics not found. Run the metrics cell first."
m = per_protein_metrics.copy()

# Ensure R^2 exists
if "r2" not in m.columns:
    m["r2"] = np.square(m["pearson_r"])

def summarize_thresholds(series: pd.Series, thresholds, direction: str, metric_name: str):
    """
    direction: 'gt' for series > thr, 'lt' for series < thr
    Returns list of dict rows with count and percent (NaNs excluded).
    """
    s = series.astype(float)
    n = int(s.notna().sum())
    rows = []
    for thr in thresholds:
        if direction == "gt":
            mask = s > thr
            sign = ">"
        elif direction == "lt":
            mask = s < thr
            sign = "<"
        else:
            raise ValueError('direction must be "gt" or "lt"')
        cnt = int(mask.sum(skipna=True))
        pct = (cnt / n * 100.0) if n else np.nan
        rows.append({
            "metric": metric_name,
            "threshold": float(thr),
            "direction": sign,
            "count": cnt,
            "percent": pct,
            "n": n
        })
    return rows

rows = []
rows += summarize_thresholds(m["cosine"], [0.95, 0.96, 0.97, 0.98], "gt", "cosine")
rows += summarize_thresholds(m["r2"],     [0.3, 0.4, 0.5, 0.6], "gt", "r2")
rows += summarize_thresholds(m["mse"],    [0.05, 0.04, 0.03, 0.02, 0.015, 0.01], "lt", "mse")

results = pd.DataFrame(rows)

# Order rows nicely: cosine, r2, mse; show '>' before '<'; thresholds ascending within each metric/direction
results["metric"] = pd.Categorical(results["metric"], categories=["cosine", "r2", "mse"], ordered=True)
results = results.sort_values(["metric", "direction", "threshold"], ascending=[True, False, True]).reset_index(drop=True)

# Round for a clean look and add a readable threshold label
results["percent"] = results["percent"].round(1)
results["threshold"] = results["threshold"].round(3)
results["rule"] = results["direction"] + " " + results["threshold"].astype(str)
results = results[["metric", "rule", "count", "percent", "n"]].rename(columns={"percent": "percent (%)"})

# Display the final result table
display(results)

# Also print means across proteins for the main metrics
means = m[["pearson_r", "cosine", "mse", "r2"]].mean(skipna=True)
print("Means across proteins:")
print(f"  Pearson r:        {means['pearson_r']:.3f}")
print(f"  Cosine similarity:{means['cosine']:.3f}")
print(f"  MSE:              {means['mse']:.6g}")
print(f"  R^2:              {means['r2']:.3f}")


### Samples


In [ ]:
# ---------- Per-sample metrics (across proteins) on normalized values ----------
# Rows = LOO samples, Cols = metrics [mse, r2, cosine]
need = {"LOO sample", "protein", "true_mm", "pred_mm"}
missing = need - set(df.columns)
assert not missing, f"df is missing required columns: {sorted(missing)}"

# pivot to samples x proteins matrices
T = df.pivot(index="LOO sample", columns="protein", values="true_mm").astype(float)
P = df.pivot(index="LOO sample", columns="protein", values="pred_mm").astype(float)

# align protein columns
common = T.columns.intersection(P.columns)
T = T[common]
P = P[common]

samples = T.index.to_numpy()
t = T.to_numpy(dtype=float)  # shape: [n_samples, n_proteins]
p = P.to_numpy(dtype=float)

# paired mask across proteins within each sample
mask = ~(np.isnan(t) | np.isnan(p))

# --- MSE per sample (across proteins) ---
sqerr = np.where(mask, (t - p) ** 2, np.nan)
mse = np.nanmean(sqerr, axis=1)

# --- Cosine similarity per sample (uncentered; across proteins) ---
num_c = np.nansum(np.where(mask, t * p, 0.0), axis=1)
t_norm = np.sqrt(np.nansum(np.where(mask, t * t, 0.0), axis=1))
p_norm = np.sqrt(np.nansum(np.where(mask, p * p, 0.0), axis=1))
den_c = t_norm * p_norm
cosine = np.divide(num_c, den_c, out=np.full_like(num_c, np.nan), where=den_c != 0)

# --- Pearson r and R^2 per sample (pairwise across proteins) ---
t_pair = np.where(mask, t, np.nan)
p_pair = np.where(mask, p, np.nan)
t_mean = np.nanmean(t_pair, axis=1, keepdims=True)
p_mean = np.nanmean(p_pair, axis=1, keepdims=True)

t_c = np.where(mask, t_pair - t_mean, 0.0)
p_c = np.where(mask, p_pair - p_mean, 0.0)

num_r = np.nansum(t_c * p_c, axis=1)
den_r = np.sqrt(np.nansum(t_c**2, axis=1) * np.nansum(p_c**2, axis=1))
pearson_r = np.divide(num_r, den_r, out=np.full_like(num_r, np.nan), where=den_r != 0)
r2 = np.square(pearson_r)

# --- assemble results ---
per_sample_metrics = pd.DataFrame(
    {"mse": mse, "r2": r2, "cosine": cosine},
    index=T.index
)

# also provide a long (stacked) version if we want metrics as rows too
per_sample_metrics_long = per_sample_metrics.stack().to_frame("value")
per_sample_metrics_long.index.set_names(["LOO sample", "metric"], inplace=True)

# show a peek and save
display(per_sample_metrics.head())
display(per_sample_metrics_long.head(9))

per_sample_metrics.to_csv("per_sample_metrics_mse_r2_cosine.csv")
print("Saved: per_sample_metrics_mse_r2_cosine.csv")

In [ ]:
# Summary counts/percentages for per-sample cosine, R^2, and MSE thresholds + metric means
assert "per_sample_metrics" in globals(), "per_sample_metrics not found. Run the per-sample metrics cell first."
m = per_sample_metrics.copy()

# Ensure required metrics exist
if "r2" not in m.columns:
    if "pearson_r" in m.columns:
        m["r2"] = np.square(m["pearson_r"])
    else:
        raise AssertionError("per_sample_metrics must include 'r2' or 'pearson_r' to derive R^2.")
for col in ("mse", "cosine", "r2"):
    assert col in m.columns, f"per_sample_metrics is missing required column '{col}'."

def summarize_thresholds(series: pd.Series, thresholds, direction: str, metric_name: str):
    """
    Compute counts/percentages across SAMPLES:
      - 'gt' counts s > thr
      - 'lt' counts s < thr
    NaNs are excluded from n.
    """
    s = series.astype(float)
    n = int(s.notna().sum())
    rows = []
    for thr in thresholds:
        if direction == "gt":
            mask = s > thr
            sign = ">"
        elif direction == "lt":
            mask = s < thr
            sign = "<"
        else:
            raise ValueError('direction must be "gt" or "lt"')
        cnt = int(mask.sum(skipna=True))
        pct = (cnt / n * 100.0) if n else np.nan
        rows.append({
            "metric": metric_name,
            "threshold": float(thr),
            "direction": sign,
            "count": cnt,
            "percent": pct,
            "n_samples": n
        })
    return rows

rows = []
rows += summarize_thresholds(m["cosine"], [0.7, 0.8, 0.9], "gt", "cosine")
rows += summarize_thresholds(m["r2"],     [0.7, 0.8, 0.9], "gt", "r2")
rows += summarize_thresholds(m["mse"],    [0.05, 0.04, 0.03, 0.02, 0.01], "lt", "mse")

results = pd.DataFrame(rows)

# Order rows: cosine, r2, mse; show '>' before '<'; thresholds ascending
results["metric"] = pd.Categorical(results["metric"], categories=["cosine", "r2", "mse"], ordered=True)
results = results.sort_values(["metric", "direction", "threshold"], ascending=[True, False, True]).reset_index(drop=True)

# Round & pretty rule label
results["percent"] = results["percent"].round(1)
results["threshold"] = results["threshold"].round(3)
results["rule"] = results["direction"] + " " + results["threshold"].astype(str)
results = results[["metric", "rule", "count", "percent", "n_samples"]].rename(columns={"percent": "percent (%)"})

# Display the threshold summary
display(results)

# Means across SAMPLES for the main metrics
means = m[["mse", "r2", "cosine"]].mean(skipna=True)
means_df = pd.DataFrame([means]).rename(columns={"r2": "R^2", "cosine": "cosine", "mse": "MSE"})
print(f"Means across samples (n={len(m)}):")
print(f"  MSE:              {means['mse']:.6g}")
print(f"  R^2:              {means['r2']:.3f}")
print(f"  Cosine similarity:{means['cosine']:.3f}")
display(means_df)


##  Protein-wise analysis


Compute per-protein metrics

In [ ]:
combined_df

In [ ]:

# Compute absolute and squared errors
combined_df['abs_error'] = (combined_df['true'] - combined_df['predicted']).abs()
combined_df['squared_error'] = (combined_df['true'] - combined_df['predicted'])**2
combined_df['rel_error']     = combined_df['abs_error'] / (combined_df['true'].abs() + eps)
# Add overall Pearson and cosine similarity as columns
corr = combined_df['true'].corr(combined_df['predicted'])
cos  = combined_df['true'].mul(combined_df['predicted']).sum() / ((combined_df['true'].pow(2).sum()**0.5) * (combined_df['predicted'].pow(2).sum()**0.5) + eps)
combined_df['pearson_corr'] = corr
combined_df['cosine_similarity'] = cos

combined_df.head()
# NEW: overall R² (same constant on all rows for convenience)
r2_overall = r2_score(combined_df['true'].values, combined_df['predicted'].values)
combined_df['r2_overall'] = r2_overall

combined_df.head()


In [ ]:


# Per-protein R² (across all LOO samples for that protein)
def _r2_group(g):
    y, yhat = g['true'].values, g['predicted'].values
    return r2_score(y, yhat) if (len(y) > 1 and np.var(y) > 0) else np.nan

r2_by_protein = (combined_df
                 .groupby('protein', dropna=False)
                 .apply(_r2_group)
                 .rename('r2')
                 .reset_index())

print("R² by protein — summary:")
print(r2_by_protein['r2'].describe())

# Histogram
plt.figure()
plt.hist(r2_by_protein['r2'].dropna().values, bins=30)
plt.xlabel('R² (per protein)')
plt.ylabel('Count')
plt.title('Distribution of R² across proteins')
plt.tight_layout()
plt.show()

# Optional: extremes
print("\nTop 10 proteins by R²:")
display(r2_by_protein.sort_values('r2', ascending=False).head(10))
print("\nBottom 10 proteins by R²:")
display(r2_by_protein.sort_values('r2', ascending=True).head(10))


In [ ]:


# Per-sample R² (across proteins within each LOO sample)
def _r2_sample_group(g):
    y, yhat = g['true'].values, g['predicted'].values
    return r2_score(y, yhat) if (len(y) > 1 and np.var(y) > 0) else np.nan

r2_by_sample = (combined_df
                .groupby('LOO sample', dropna=False)
                .apply(_r2_sample_group)
                .rename('r2')
                .reset_index())

print("R² by sample — summary:")
print(r2_by_sample['r2'].describe())

# Histogram
plt.figure()
plt.hist(r2_by_sample['r2'].dropna().values, bins=30)
plt.xlabel('R² (per LOO sample)')
plt.ylabel('Count')
plt.title('Distribution of R² across LOO samples')
plt.tight_layout()
plt.show()

# Optional: extremes
print("\nTop 10 LOO samples by R²:")
display(r2_by_sample.sort_values('r2', ascending=False).head(10))
print("\nBottom 10 LOO samples by R²:")
display(r2_by_sample.sort_values('r2', ascending=True).head(10))


In [ ]:
# pull clean squared errors
se = (pd.to_numeric(combined_df["squared_error"], errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .dropna())

# choose an x-axis cap (tweak if you like)
p95, p99, p999 = np.percentile(se, [95, 99, 99.9])
xmax = p99          # show up to the 99th percentile
tail_n = int((se > xmax).sum())

fig, ax = plt.subplots(figsize=(8, 5), dpi=130)

# histogram only for values within the cap
data_in = se[se <= xmax]
bins = np.linspace(0, xmax, 60)
ax.hist(data_in, bins=bins)

# annotate mean/median (only if they fall within the shown range)
mean_val = se.mean()
med_val  = se.median()
if med_val <= xmax:
    ax.axvline(med_val, linestyle="--", linewidth=1.5, label=f"median = {med_val:.4g}")
else:
    ax.text(xmax, ax.get_ylim()[1]*0.90, f"median = {med_val:.4g} (off-scale)",
            ha="right", va="top", fontsize=9)

if mean_val <= xmax:
    ax.axvline(mean_val, linestyle="-", linewidth=1.5, label=f"mean = {mean_val:.4g}")
else:
    ax.text(xmax, ax.get_ylim()[1]*0.80, f"mean = {mean_val:.4g} (off-scale)",
            ha="right", va="top", fontsize=9)

# x-axis limits and a note about the hidden tail
ax.set_xlim(0, xmax)
if tail_n > 0:
    ax.text(xmax, ax.get_ylim()[1]*0.70, f">{xmax:.3g}: {tail_n} outliers",
            ha="right", va="top", fontsize=9)

ax.set_title("Distribution of Squared Errors")
ax.set_xlabel("Squared error")
ax.set_ylabel("Count")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()


In [ ]:
# Per-sample metrics from combined_df → samples_df (369 rows expected)
eps = 1e-12

combined_df = combined_df.copy()

def _cos_sim(a, b):
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    denom = na * nb
    return float(np.dot(a, b) / denom) if denom > 0 else np.nan

# the column holding sample IDs (handles the space in "LOO sample")
sample_col = 'LOO sample' if 'LOO sample' in combined_df.columns else 'sample'

samples_df = (
    combined_df.groupby(sample_col, sort=False)
      .apply(lambda g: pd.Series({
          'n_proteins': len(g),
          'mae': g['abs_error'].mean(),
          'mse': g['squared_error'].mean(),
          'rmse': np.sqrt(g['squared_error'].mean()),
          'cosine_similarity': _cos_sim(g['true'].to_numpy(), g['predicted'].to_numpy()),
          'pearson_corr': g['true'].corr(g['predicted'])
      }))
      .reset_index()
      .rename(columns={sample_col: 'sample'})
)

samples_df.head()


In [ ]:
#  tissue counts from samples_df 

min_n = 369 # threshold for "low-representation" tissues

# Tissue is the suffix after the last underscore
samples_df = samples_df.copy()
samples_df["tissue"] = samples_df["sample"].astype(str).str.rsplit("_", n=1).str[-1]

# Count unique samples per tissue
tissue_counts = (
    samples_df.drop_duplicates("sample")
              .groupby("tissue")["sample"]
              .nunique()
              .sort_values()
)

low_tissues = tissue_counts[tissue_counts <= min_n]

print(f"Total tissues: {tissue_counts.size}")
print(f"Tissues with <= {min_n} samples:")
if low_tissues.empty:
    print("  (none)")
else:
    # pretty print as two columns
    print(low_tissues.to_string())


In [ ]:
#  filter combined_df by tissue representation 
n = 18  # set the threshold here

# Reuse tissue mapping from Cell 1 (samples_df already has a 'tissue' column)
tissue_sample_counts = (
    samples_df.drop_duplicates("sample")
              .groupby("tissue")["sample"]
              .nunique()
)

low_tissues = set(tissue_sample_counts[tissue_sample_counts < n].index)

# All samples that belong to low-representation tissues
low_tissue_samples = set(
    samples_df.loc[samples_df["tissue"].isin(low_tissues), "sample"].unique()
)

# combined_df may name the sample column as 'LOO_sample' or 'LOO sample'—we handle both.
sample_col = "LOO_sample" if "LOO_sample" in combined_df.columns else (
    "LOO sample" if "LOO sample" in combined_df.columns else "sample"
)

before = len(combined_df)
clean = combined_df.loc[~combined_df[sample_col].isin(low_tissue_samples)].copy()
after = len(clean)

print(f"Threshold n = {n}")
print(f"Low-representation tissues ({len(low_tissues)}): {sorted(low_tissues) if low_tissues else 'none'}")
print(f"Removed {before - after} rows; kept {after} rows.")


In [ ]:
def _cos_sim(a, b):
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    denom = na * nb
    return float(np.dot(a, b) / denom) if denom > 0 else np.nan

per_protein = (
    clean.groupby('protein', sort=False)
    .apply(lambda g: pd.Series({
        'n_samples': len(g),
        'mse': g['squared_error'].mean(),
        'cosine_similarity': _cos_sim(g['true'].to_numpy(), g['predicted'].to_numpy()),
        'pearson_corr': g['true'].corr(g['predicted'])
    }))
    .reset_index()
)

### Top 100 proteins by mean squared error-cosine similarity-pearson correlation

In [ ]:
# Per-protein ranking tables: MAE, cosine similarity, Pearson correlation


top100_mae     = per_protein.sort_values('mse', ascending=True).head(100)         # lower is better
top100_cosine  = per_protein.sort_values('cosine_similarity', ascending=False).head(100)
top100_pearson = per_protein.sort_values('pearson_corr', ascending=False).head(100)

print("Top 100 proteins by lowest MAE")
display(top100_mae)

print("\nTop 100 proteins by highest cosine similarity")
display(top100_cosine)

print("\nTop 100 proteins by highest Pearson correlation")
display(top100_pearson)

In [ ]:
# Intersection
set_mae  = set(top100_mae['protein'])
set_cos  = set(top100_cosine['protein'])
set_pear = set(top100_pearson['protein'])
inter_prots = sorted(set_mae & set_cos & set_pear)

print(f"Proteins in the intersection of all three top-100 lists: {len(inter_prots)}")

if len(inter_prots) == 0:
    print("No overlap across the three lists.")
else:
    # Add ranks for context
    pp_idx = per_protein.set_index('protein')
    ranked = pp_idx.assign(
        mae_rank=pp_idx['mae'].rank(method='min', ascending=True).astype(int),
        cosine_rank=pp_idx['cosine_similarity'].rank(method='min', ascending=False).astype(int),
        pearson_rank=pp_idx['pearson_corr'].rank(method='min', ascending=False).astype(int),
    )

    intersection_table = (
        ranked.loc[inter_prots, ['n_samples', 'mae', 'mae_rank',
                                 'cosine_similarity', 'cosine_rank',
                                 'pearson_corr', 'pearson_rank']]
        .sort_values(['mae', 'cosine_rank', 'pearson_rank'])
        .reset_index()
        .rename(columns={'protein': 'protein'})
    )

    display(intersection_table)

### Error Distributions and thresholds 

#### means

In [ ]:
# Compute mean squared error for each protein across samples
protein_means = combined_df.groupby("protein")["squared_error"].mean().reset_index()
protein_means.rename(columns={"squared_error": "mean_squared_error"}, inplace=True)

# Plot the distribution
plt.figure(figsize=(10, 5))
sns.histplot(protein_means["mean_squared_error"], bins=30, kde=True, color="steelblue")

plt.title("Distribution of Mean Squared Errors Across Proteins")
plt.xlabel("Mean Squared Error")
plt.ylabel("Number of Proteins")
plt.tight_layout()
plt.show()

In [ ]:
# Compute mean squared error for each protein across samples
protein_means = clean.groupby("protein")["squared_error"].mean().reset_index()
protein_means.rename(columns={"squared_error": "mean_squared_error"}, inplace=True)

# Plot the distribution
plt.figure(figsize=(10, 5))
sns.histplot(protein_means["mean_squared_error"], bins=30, kde=True, color="steelblue")

plt.title("Distribution of Mean Squared Errors Across Proteins")
plt.xlabel("Mean Squared Error")
plt.ylabel("Number of Proteins")
plt.tight_layout()
plt.show()

In [ ]:
# MAE < 0.1
threshold = 0.1

# mask + subset for plotting
mask = protein_means["mean_squared_error"] < threshold
subset = protein_means.loc[mask, "mean_squared_error"]

print(f"Proteins with mean_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    # --- histogram (seaborn) ---
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    plt.title(f"Distribution of Mean Squared Errors (MSE < {threshold})")
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    plt.show()

    # --- table of proteins with MSE < threshold ---
    # If there's a 'protein' column, use it; otherwise, lift the index into a column.
    if "protein" in protein_means.columns:
        table = protein_means.loc[mask, ["protein", "mean_squared_error"]].copy()
    else:
        table = (
            protein_means.loc[mask, ["mean_squared_error"]]
            .reset_index()
            .rename(columns={"index": "protein"})
        )

    # sort for readability and format
    table = table.sort_values("mean_squared_error").reset_index(drop=True)
    table["mean_squared_error"] = table["mean_squared_error"].astype(float)

    # Display nicely in notebooks (falls back to print if display isn't available)
    try:
        from IPython.display import display
        print("\nTable: Proteins with MSE < {:.3f} (sorted ascending)".format(threshold))
        display(table.style.format({"mean_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Proteins with MSE < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
# Cell: MAE < 0.2
threshold = 0.2
subset = protein_means.loc[protein_means["mean_squared_error"] < threshold, "mean_squared_error"]

print(f"Proteins with mean_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    plt.title("Distribution of Mean Squared Errors (MSE < 0.2)")
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    plt.show()


In [ ]:
# Cell: MAE < 0.3
threshold = 0.3
subset = protein_means.loc[protein_means["mean_squared_error"] < threshold, "mean_squared_error"]

print(f"Proteins with mean_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    plt.title("Distribution of Mean Squared Errors (MSE < 0.3)")
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    plt.show()


#### medians

In [ ]:
# Compute median squared error for each protein across samples
protein_medians = combined_df.groupby("protein")["squared_error"].median().reset_index()
protein_medians.rename(columns={"squared_error": "median_squared_error"}, inplace=True)

# Plot the distribution
plt.figure(figsize=(10, 5))
sns.histplot(protein_medians["median_squared_error"], bins=30, kde=True, color="steelblue")
plt.title("Distribution of Median Squared Errors Across Proteins")
plt.xlabel("Median Squared Error")
plt.ylabel("Number of Proteins")
plt.tight_layout()
plt.show()

In [ ]:
# MAE < 0.1
threshold = 0.05

# mask + subset for plotting
mask = protein_medians["median_squared_error"] < threshold
subset = protein_medians.loc[mask, "median_squared_error"]

print(f"Proteins with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    # --- histogram (seaborn) ---
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    plt.title(f"Distribution of Median Squared Errors < {threshold}")
    plt.xlabel("Median Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    plt.show()

    # --- table of proteins with MSE < threshold ---
    # If there's a 'protein' column, use it; otherwise, lift the index into a column.
    if "protein" in protein_medians.columns:
        table = protein_medians.loc[mask, ["protein", "median_squared_error"]].copy()
    else:
        table = (
            protein_medians.loc[mask, ["median_squared_error"]]
            .reset_index()
            .rename(columns={"index": "protein"})
        )

    # sort for readability and format
    table = table.sort_values("median_squared_error").reset_index(drop=True)
    table["median_squared_error"] = table["median_squared_error"].astype(float)

    # Display nicely in notebooks (falls back to print if display isn't available)
    try:
        from IPython.display import display
        print("\nTable: Proteins with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        display(table.style.format({"median_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Proteins with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))

In [ ]:
# MAE < 0.1
threshold = 0.1

# mask + subset for plotting
mask = protein_medians["median_squared_error"] < threshold
subset = protein_medians.loc[mask, "median_squared_error"]

print(f"Proteins with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    # --- histogram (seaborn) ---
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    plt.title(f"Distribution of Median Squared Errors < {threshold}")
    plt.xlabel("Median Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    plt.show()

    # --- table of proteins with MSE < threshold ---
    # If there's a 'protein' column, use it; otherwise, lift the index into a column.
    if "protein" in protein_medians.columns:
        table = protein_medians.loc[mask, ["protein", "median_squared_error"]].copy()
    else:
        table = (
            protein_medians.loc[mask, ["median_squared_error"]]
            .reset_index()
            .rename(columns={"index": "protein"})
        )

    # sort for readability and format
    table = table.sort_values("median_squared_error").reset_index(drop=True)
    table["median_squared_error"] = table["median_squared_error"].astype(float)

    # Display nicely in notebooks (falls back to print if display isn't available)
    try:
        from IPython.display import display
        print("\nTable: Proteins with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        display(table.style.format({"median_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Proteins with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
enet01 = set(list(table['protein']))
print(len(enet01))

In [ ]:
enet01

In [ ]:
# XGBoost proteins passing the same rule (median squared error < 0.1).
# Recomputed from the merged XGB LOO results (04. xgb/model_results -> xgb_res/xgb_all_results.h5),
# so the legacy hand-made res_data/proteins01.csv is no longer required (used only if the H5 is absent).
from pathlib import Path
xgb_h5 = Path('../data/xgb_all_results.h5')
if xgb_h5.exists():
    with pd.HDFStore(xgb_h5, mode='r') as _st:
        _xgb = pd.concat([_st[k] for k in _st.keys()], ignore_index=True)
    _sq = (pd.to_numeric(_xgb['true'], errors='coerce') - pd.to_numeric(_xgb['predicted'], errors='coerce')) ** 2
    _med = _sq.groupby(_xgb['protein'].astype(str)).median()
    xgb01 = set(_med.index[_med < 0.1])
elif Path('../data/proteins01.csv').exists():
    xgb01 = set(pd.read_csv('../data/proteins01.csv')['protein'].astype(str))
else:
    xgb01 = set()
    print('No XGB results found - ENet/XGB overlap below is skipped.')
print(len(xgb01))

In [ ]:
len(enet01 & xgb01)

In [ ]:
len(enet01 | xgb01)

In [ ]:
# Cell: MAE < 0.2
threshold = 0.2
subset = protein_medians.loc[protein_medians["median_squared_error"] < threshold, "median_squared_error"]

print(f"Proteins with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    plt.title("Distribution of Median Squared Errors (MSE < 0.2)")
    plt.xlabel("Median Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell: MAE < 0.3
threshold = 0.3
subset = protein_medians.loc[protein_medians["median_squared_error"] < threshold, "median_squared_error"]

print(f"Proteins with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No proteins below this threshold to plot.")
else:
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="steelblue")
    plt.title("Distribution of Median Squared Errors (MSE < 0.3)")
    plt.xlabel("Median Squared Error")
    plt.ylabel("Number of Proteins")
    plt.tight_layout()
    plt.show()


### Other per-protein plots

In [ ]:
# MSE vs expression with robust limits and readable 1:1 line

# Recompute per-protein summary (safe if it already exists)
prot_perf = (
    combined_df.groupby('protein')
    .agg(expression=('true', 'mean'),
         mse=('squared_error', 'mean'),
         mae=('abs_error', 'mean'))
    .reset_index()
)

x = prot_perf['expression'].to_numpy()
y = prot_perf['mse'].to_numpy()

# Robust axis limits (clip to 0.5–99.5 percentiles and pad 5%)
x_lo, x_hi = np.nanpercentile(x, [0.5, 99.5])
y_lo, y_hi = np.nanpercentile(y, [0.5, 99.5])
x_pad = 0.05 * (x_hi - x_lo) if np.isfinite(x_hi - x_lo) else 1.0
y_pad = 0.05 * (y_hi - y_lo) if np.isfinite(y_hi - y_lo) else 1.0
xlim = (x_lo - x_pad, x_hi + x_pad)
ylim = (y_lo - y_pad, y_hi + y_pad)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=prot_perf, x='expression', y='mse',
    s=18, alpha=0.5, linewidth=0, edgecolor=None
)


plt.xlim(*xlim)
plt.ylim(*ylim)
plt.xlabel('Mean expression (true)')
plt.ylabel('Mean squared error (MSE)')
plt.title('Protein-wise: MSE vs Expression')
plt.grid(True, linestyle=':', linewidth=0.7, alpha=0.6)
plt.tight_layout()
plt.show()


In [ ]:
# Mean and variance per protein
protein_stats = combined_df.groupby('protein')['abs_error'].agg(['mean', 'var']).reset_index()

# Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(data=protein_stats, x='mean', y='var')
plt.title("Mean vs Variance of Absolute Error per Protein")
plt.xlabel("Mean Absolute Error")
plt.ylabel("Variance of Absolute Error")
plt.grid(True)
plt.show()


#### mean error vs variance of true values

In [ ]:
#Aggregate per protein 

prot_stats = (
    combined_df
    .groupby('protein')
    .agg(
        mean_error = ('abs_error', 'mean'),
        var_true   = ('true', 'var'),
    )
    .reset_index()
)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=prot_stats,
    x='var_true',
    y='mean_error',
    s=25,
    edgecolor='k'
)
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.title('Protein‑wise: Mean Prediction Error vs Variance of True Values')
plt.xlabel('Variance of True Values')
plt.ylabel('Mean Prediction Error')
plt.tight_layout()
plt.show()

### Cosine similaity 

In [ ]:
# Top 50, vertical bars, steelblue gradient
# prep
df = per_protein.copy()
if 'protein' in df.columns:
    df = df.set_index('protein')

df_top = (
    df[['cosine_similarity']]
    .dropna()
    .sort_values('cosine_similarity', ascending=False)
    .head(50)
)

vals   = df_top['cosine_similarity'].to_numpy()
labels = df_top.index.astype(str)

# custom steelblue gradient (deep navy → steelblue → pale)
steelblue_cmap = LinearSegmentedColormap.from_list(
    "steelblue_ramp",
    ["#0f2d45", "#1f4e79", "#4682B4", "#9ab6d1", "#e2eef7"]
)

# color by rank so the gradient flows left→right
colors = steelblue_cmap(np.linspace(0.07, 0.97, len(vals)))

fig, ax = plt.subplots(figsize=(18, 6), dpi=140)

x = np.arange(len(vals))
ax.bar(x, vals, color=colors, width=0.85)

# labels & limits
ax.set_title('Top 50 proteins by cosine similarity', pad=12)
ax.set_ylabel('Cosine similarity')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=90, fontsize=12)
ax.set_ylim(0, max(1.0, vals.max() + 0.05))

# subtle grid & cleanup
ax.grid(False, which='both')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()


How many prteins after applying cosine similarity threshold of 0.5, 0.6, 0.7, 0.8, 0.9

In [ ]:
# Summary table for cosine similarity thresholds + save per-threshold protein lists

from datetime import datetime

df = per_protein.copy()
if 'protein' in df.columns:
    df = df.set_index('protein')

# restrict to non-null cosine similarities
cs = df['cosine_similarity'].dropna()
total = int(cs.shape[0])

thresholds = [0.5, 0.6, 0.7, 0.8]
rows = []

# prepare output dir + timestamp
out_dir = Path("exports")
out_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now().strftime("%Y%m%d-%H%M%S")

for t in thresholds:
    mask = cs > t
    n = int(mask.sum())
    pct = (n / total * 100) if total else 0.0
    rows.append({"threshold": f">{t:.1f}", "n_proteins": n, "% of total": f"{pct:.1f}%"})

    # build table for proteins above this threshold
    tbl = (
        df.loc[mask, ["cosine_similarity"]]
          .sort_values("cosine_similarity", ascending=False)
          .assign(protein=lambda d: d.index)
          .loc[:, ["protein", "cosine_similarity"]]
    )

    # save CSV (empty files will still be created with headers)
    fname = out_dir / f"proteins_cosine_similarity_gt_{str(t).replace('.', 'p')}_{stamp}.csv"
    tbl.to_csv(fname, index=False)
    print(f"Saved {n} proteins with cosine_similarity > {t:.1f} -> {fname.resolve()}")

summary = pd.DataFrame(rows).set_index("threshold")

print(f"\nTotal proteins considered: {total}\n")
print(summary.to_string())


In [ ]:
#positive 
pos = df[df['cosine_similarity'] > 0]
len(pos)

In [ ]:


# total rows considered (robust to NaNs)
total = int(pos['cosine_similarity'].notna().sum())

thresholds = [0.5, 0.6, 0.7, 0.8]
rows = []
for t in thresholds:
    n = int((pos['cosine_similarity'] > t).sum())
    pct = (n / total * 100) if total else 0.0
    rows.append({"threshold": f">{t:.1f}", "n_proteins": n, "% of total": f"{pct:.1f}%"})

summary = pd.DataFrame(rows).set_index("threshold")

print(f"Total proteins considered: {total}\n")
print(summary.to_string())


In [ ]:
df[['cosine_similarity']].describe()

### Comparison with baseline #to do

In [ ]:
# Placeholder for a future baseline comparison (section marked 'to do').
# pe85.csv is not produced by the pipeline and is not used anywhere below; loaded only if present.
from pathlib import Path
pe85 = pd.read_csv('../data/pe85.csv', dtype=str)['id'].tolist() if Path('../data/pe85.csv').exists() else []

# Samples analysis


### Top 100 samples by mean squared error, cosine similarity, pearson correlation

In [ ]:
# Per-sample metrics from combined_df → samples_df (369 rows expected)
eps = 1e-12

combined_df = combined_df.copy()

# row-wise errors
combined_df['abs_error']     = (combined_df['true'] - combined_df['predicted']).abs()
combined_df['squared_error'] = (combined_df['true'] - combined_df['predicted'])**2
combined_df['rel_error']     = combined_df['abs_error'] / (combined_df['true'].abs() + eps)

def _cos_sim(a, b):
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    denom = na * nb
    return float(np.dot(a, b) / denom) if denom > 0 else np.nan

# the column holding sample IDs (handles the space in "LOO sample")
sample_col = 'LOO sample' if 'LOO sample' in combined_df.columns else 'sample'

samples_df = (
    combined_df.groupby(sample_col, sort=False)
      .apply(lambda g: pd.Series({
          'n_proteins': len(g),
          'mae': g['abs_error'].mean(),
          'mse': g['squared_error'].mean(),
          'rmse': np.sqrt(g['squared_error'].mean()),
          'cosine_similarity': _cos_sim(g['true'].to_numpy(), g['predicted'].to_numpy()),
          'pearson_corr': g['true'].corr(g['predicted'])
      }))
      .reset_index()
      .rename(columns={sample_col: 'sample'})
)

samples_df.head()


### Top 100 proteins by mean squared error, cosine similarity, pearson correlation

In [ ]:

top100_mse     = samples_df.sort_values('mse', ascending=True).head(100)               # lower is better
top100_mae     = samples_df.sort_values('mae', ascending=True).head(100)               # optional: also show by MAE
top100_cosine  = samples_df.sort_values('cosine_similarity', ascending=False).head(100)
top100_pearson = samples_df.sort_values('pearson_corr',       ascending=False).head(100)

print("Top 100 samples by lowest MSE")
display(top100_mse)

print("\nTop 100 samples by lowest MAE")
display(top100_mae)

print("\nTop 100 samples by highest cosine similarity")
display(top100_cosine)

print("\nTop 100 samples by highest Pearson correlation")
display(top100_pearson)


In [ ]:
sample_error_summary = combined_df.groupby('LOO sample').agg(
    mean_abs_error=('abs_error', 'mean'),
    std_abs_error=('abs_error', 'std'),
    mean_squared_error=('squared_error', 'mean'),
    std_squared_error=('squared_error', 'std'),
    count=('LOO sample', 'count')
).sort_values(by='mean_abs_error', ascending=False)

sample_error_summary.head()

### MSE thresholding and distributions

#### means

In [ ]:
# Compute median absolute error across proteins for each sample
sample_means = combined_df.groupby("LOO sample")["squared_error"].mean().reset_index()
sample_means.rename(columns={"squared_error": "mean_squared_error"}, inplace=True)

# Plot
plt.figure(figsize=(10, 5))
sns.histplot(sample_means["mean_squared_error"], bins=30, kde=True, color="mediumseagreen")
plt.title("Distribution of Mean Squared Errors Across Samples")
plt.xlabel("Mean Squared Error")
plt.ylabel("Number of Samples")
plt.tight_layout()
plt.show()


In [ ]:
# Mean Squared Error per sample + threshold plot (< 0.1)

# column holding sample IDs (handles the space in "LOO sample")
sample_col = "LOO sample" if "LOO sample" in combined_df.columns else "sample"

# mean squared error across proteins for each sample
sample_means = (
    combined_df.groupby(sample_col)["squared_error"]
               .mean()
               .reset_index()
               .rename(columns={sample_col: "sample", "squared_error": "mean_squared_error"})
)

threshold = 0.1

# mask + subset for plotting
mask = sample_means["mean_squared_error"] < threshold
subset = sample_means.loc[mask, "mean_squared_error"]

print(f"Samples with mean_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    # --- histogram (seaborn) ---
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    plt.title(f"Distribution of Sample Mean Squared Errors (MSE < {threshold})")
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    plt.show()

    # --- table of samples with MSE < threshold ---
    table = (
        sample_means.loc[mask, ["sample", "mean_squared_error"]]
        .sort_values("mean_squared_error")
        .reset_index(drop=True)
    )

    try:
        from IPython.display import display
        print("\nTable: Samples with MSE < {:.3f} (sorted ascending)".format(threshold))
        display(table.style.format({"mean_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with MSE < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
threshold = 0.2

mask = sample_means["mean_squared_error"] < threshold
subset = sample_means.loc[mask, "mean_squared_error"]

print(f"Samples with mean_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    plt.title(f"Distribution of Sample Mean Squared Errors (MSE < {threshold})")
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    plt.show()

    table = (
        sample_means.loc[mask, ["sample", "mean_squared_error"]]
        .sort_values("mean_squared_error")
        .reset_index(drop=True)
    )
    try:
        from IPython.display import display
        print("\nTable: Samples with MSE < {:.3f} (sorted ascending)".format(threshold))
        #display(table.style.format({"mean_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with MSE < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
threshold = 0.3

mask = sample_means["mean_squared_error"] < threshold
subset = sample_means.loc[mask, "mean_squared_error"]

print(f"Samples with mean_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    plt.title(f"Distribution of Sample Mean Squared Errors (MSE < {threshold})")
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    plt.show()

    table = (
        sample_means.loc[mask, ["sample", "mean_squared_error"]]
        .sort_values("mean_squared_error")
        .reset_index(drop=True)
    )
    try:
        from IPython.display import display
        print("\nTable: Samples with MSE < {:.3f} (sorted ascending)".format(threshold))
        #display(table.style.format({"mean_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with MSE < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


#### Medians

In [ ]:
# Compute median absolute error across proteins for each sample
sample_medians = combined_df.groupby("LOO sample")["squared_error"].median().reset_index()
sample_medians.rename(columns={"squared_error": "median_squared_error"}, inplace=True)

# Plot
plt.figure(figsize=(10, 5))
sns.histplot(sample_medians["median_squared_error"], bins=30, kde=True, color="mediumseagreen")
plt.title("Distribution of Median Squared Errors Across Samples")
plt.ylabel("Number of Samples")
plt.tight_layout()
plt.show()


In [ ]:
# Mean Squared Error per sample + threshold plot (< 0.05)

# column holding sample IDs (handles the space in "LOO sample")
sample_col = "LOO sample" if "LOO sample" in combined_df.columns else "sample"

threshold = 0.05

# mask + subset for plotting
mask = sample_medians["median_squared_error"] < threshold
subset = sample_medians.loc[mask, "median_squared_error"]

print(f"Samples with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    # --- histogram (seaborn) ---
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    plt.title(f"Distribution of Sample Mean Squared Errors (MSE < {threshold})")
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    plt.show()

    # --- table of samples with MSE < threshold ---
    table = (
        sample_medians.loc[mask, [sample_col, "median_squared_error"]]
        .sort_values("median_squared_error")
        .reset_index(drop=True)
    )

    try:
        from IPython.display import display
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        display(table.style.format({"median_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
# Mean Squared Error per sample + threshold plot (< 0.1)

# column holding sample IDs (handles the space in "LOO sample")
sample_col = "LOO sample" if "LOO sample" in combined_df.columns else "sample"

threshold = 0.1

# mask + subset for plotting
mask = sample_medians["median_squared_error"] < threshold
subset = sample_medians.loc[mask, "median_squared_error"]

print(f"Samples with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    # --- histogram (seaborn) ---
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    plt.title(f"Distribution of Sample Mean Squared Errors (MSE < {threshold})")
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    plt.show()

    # --- table of samples with MSE < threshold ---
    table = (
        sample_medians.loc[mask, [sample_col, "median_squared_error"]]
        .sort_values("median_squared_error")
        .reset_index(drop=True)
    )

    # --- SAVE TABLE ---
    from pathlib import Path
    from datetime import datetime

    out_dir = Path("exports")
    out_dir.mkdir(parents=True, exist_ok=True)

    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    base = f"samples_median_squared_error_lt_{threshold}".replace(".", "p")

    csv_path = out_dir / f"{base}_{stamp}.csv"

    table.to_csv(csv_path, index=False)
    print(f"Saved CSV -> {csv_path.resolve()}")

    # --- display nicely in notebooks ---
    try:
        from IPython.display import display
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        display(table.style.format({"median_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
threshold = 0.2

mask = sample_medians["median_squared_error"] < threshold
subset = sample_medians.loc[mask, "median_squared_error"]

print(f"Samples with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    plt.title(f"Distribution of Sample Mean Squared Errors (MSE < {threshold})")
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    plt.show()

    table = (
        sample_medians.loc[mask, [sample_col, "median_squared_error"]]
        .sort_values("median_squared_error")
        .reset_index(drop=True)
    )
    try:
        from IPython.display import display
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        #display(table.style.format({"median_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


In [ ]:
threshold = 0.3

mask = sample_medians["median_squared_error"] < threshold
subset = sample_medians.loc[mask, "median_squared_error"]

print(f"Samples with median_squared_error < {threshold}: {subset.shape[0]}")

if subset.empty:
    print("No samples below this threshold to plot.")
else:
    plt.figure(figsize=(10, 5))
    sns.histplot(subset, bins=30, kde=True, color="mediumseagreen")
    plt.title(f"Distribution of Sample Mean Squared Errors (MSE < {threshold})")
    plt.xlabel("Mean Squared Error")
    plt.ylabel("Number of Samples")
    plt.tight_layout()
    plt.show()

    table = (
        sample_medians.loc[mask, [sample_col, "median_squared_error"]]
        .sort_values("median_squared_error")
        .reset_index(drop=True)
    )
    try:
        from IPython.display import display
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        #display(table.style.format({"median_squared_error": "{:.6f}"}))
    except Exception:
        print("\nTable: Samples with Median Squared Error < {:.3f} (sorted ascending)".format(threshold))
        print(table.to_string(index=False))


### Cosine similarity

In [ ]:
# Top 50 samples by cosine similarity — vertical bars, Medium SeaGreen gradient
# Build samples_df if it doesn't already exist
try:
    samples_df
except NameError:
    sample_col = "LOO sample" if "LOO sample" in combined_df.columns else "sample"
    if "squared_error" not in combined_df.columns:
        combined_df = combined_df.copy()
        combined_df["squared_error"] = (combined_df["true"] - combined_df["predicted"])**2

    samples_df = (
        clean.groupby(sample_col, sort=False)
        .apply(lambda g: pd.Series({
            "n_proteins": len(g),
            "mse": g["squared_error"].mean(),
            "cosine_similarity": _cos_sim(g["true"].to_numpy(), g["predicted"].to_numpy()),
            "pearson_corr": g["true"].corr(g["predicted"]),
        }))
        .reset_index()
        .rename(columns={sample_col: "sample"})
    )

# Sort & take top K
k = 50
df_top = (
    samples_df[["sample", "cosine_similarity"]]
    .dropna()
    .sort_values("cosine_similarity", ascending=False)
    .head(k)
)

vals   = df_top["cosine_similarity"].to_numpy()
labels = df_top["sample"].astype(str).to_numpy()

# Medium SeaGreen tonality (deep green → MediumSeaGreen → pale mint)
seagreen_cmap = LinearSegmentedColormap.from_list(
    "seagreen_ramp",
    ["#0f4d3a", "#2f8f68", "#3CB371", "#8ad9b2", "#d7f5e8"]  # tweak if we want it cooler/warmer
)
colors = seagreen_cmap(np.linspace(0.07, 0.97, len(vals)))  # smooth rank-based gradient

fig, ax = plt.subplots(figsize=(18, 6), dpi=140)
x = np.arange(len(vals))
ax.bar(x, vals, color=colors, width=0.85)

# Labels & limits
ax.set_title("Top 50 samples by cosine similarity", pad=12)
ax.set_ylabel("Cosine similarity")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=90, fontsize=8)
ax.set_ylim(0, max(1.0, vals.max() + 0.05))

# Tidy up
ax.grid(False, which='both')
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()


In [ ]:
s = samples_df['sample'].astype(str).str.strip()
parts = s.str.split('_', n=1, expand=True)     # use n=1 as a keyword
samples_df['tissue'] = parts[1].where(parts[1].notna() & (parts[1] != ''), parts[0])

# keep one sample per tissue: the one with the highest cosine
df_best = (samples_df.dropna(subset=['tissue','cosine_similarity'])
                     .sort_values('cosine_similarity', ascending=False)
                     .drop_duplicates(subset=['tissue'], keep='first'))

# data for plotting (ALL tissues)
vals   = df_best['cosine_similarity'].to_numpy()
labels = df_best['tissue'].astype(str).to_numpy()

# Medium SeaGreen gradient (deep → light)
seagreen_cmap = LinearSegmentedColormap.from_list(
    "seagreen_ramp", ["#0f4d3a", "#2f8f68", "#3CB371", "#8ad9b2", "#d7f5e8"]
)
colors = seagreen_cmap(np.linspace(0.07, 0.97, len(vals)))

# dynamic width to fit all tissues
fig_w = max(12, 0.35 * len(vals))
fig, ax = plt.subplots(figsize=(fig_w, 6), dpi=130)

x = np.arange(len(vals))
ax.bar(x, vals, color=colors, width=0.85, edgecolor='none', linewidth=0)

ax.set_title("Best sample per tissue by cosine similarity (all tissues)", pad=10)
ax.set_ylabel("Cosine similarity")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=9)
ax.set_ylim(0, max(1.0, float(vals.max()) + 0.05))

# clean look (no grid/edges)
ax.grid(False, which='both')
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# total rows considered (robust to NaNs)
total = int(samples_df['cosine_similarity'].notna().sum())

thresholds = [0.5, 0.6, 0.7, 0.8]
rows = []
for t in thresholds:
    n = int((samples_df['cosine_similarity'] > t).sum())
    pct = (n / total * 100) if total else 0.0
    rows.append({"threshold": f">{t:.1f}", "n_samples": n, "% of total": f"{pct:.1f}%"})

summary = pd.DataFrame(rows).set_index("threshold")

print(f"Total Samples considered: {total}\n")
print(summary.to_string())


In [ ]:
# --- aggregate per tissue (MEAN)
tissue_table = (
    samples_df.dropna(subset=['tissue'])
              .groupby('tissue', as_index=False)
              .agg(n_samples=('sample', 'nunique'),
                   cosine_similarity=('cosine_similarity', 'mean'),
                   mse=('mse', 'mean'))
              .sort_values('cosine_similarity', ascending=False)
)

# --- display
try:
    display(
        tissue_table.style
            .format({'cosine_similarity': '{:.4f}', 'mse': '{:.6f}', 'n_samples': '{:d}'})
            .set_properties(subset=['tissue'], **{'font-weight': 'bold'})
    )
except Exception:
    print(tissue_table.to_string(index=False))

In [ ]:

# --- aggregate per tissue (median; switch to 'mean' if preferred)
tissue_stats = (
    samples_df.dropna(subset=['tissue', 'cosine_similarity'])
              .groupby('tissue')
              .agg(cosine_similarity=('cosine_similarity', 'median'),
                   n_samples=('sample', 'nunique'))
              .sort_values('cosine_similarity', ascending=False)
)

vals   = tissue_stats['cosine_similarity'].to_numpy()
labels = tissue_stats.index.astype(str).to_numpy()

# Light purple (lilac) gradient
lilac_cmap = LinearSegmentedColormap.from_list(
    "lilac_ramp",
    ["#5b4b8a", "#7e6db1", "#a896e0", "#cdbbf0", "#f0e9ff"] 
)
colors = lilac_cmap(np.linspace(0.07, 0.97, len(vals)))

# dynamic width for all tissues
fig_w = max(12, 0.35 * len(vals))
fig, ax = plt.subplots(figsize=(fig_w, 6), dpi=130)

x = np.arange(len(vals))
ax.bar(x, vals, color=colors, width=0.85, edgecolor='none', linewidth=0)

ax.set_title("Tissue-level cosine similarity (median across samples)", pad=10)
ax.set_ylabel("Cosine similarity")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=60, ha='right', fontsize=9)
ax.set_ylim(0, max(1.0, float(vals.max()) + 0.05))

ax.set_axisbelow(True)
ax.grid(False, which='both')

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

### Other statistical plots on Samples

In [ ]:
sample_stats = combined_df.groupby('LOO sample')['abs_error'].agg(['mean', 'var']).reset_index()

# Plot
plt.figure(figsize=(10, 6))
sns.scatterplot(data=sample_stats, x='mean', y='var')
plt.title("Mean vs Variance of Absolute Error per LOO Sample")
plt.xlabel("Mean Absolute Error")
plt.ylabel("Variance of Absolute Error")
plt.grid(True)
plt.show()


In [ ]:
# Aggregate per LOO sample
sample_stats = (
    combined_df
    .groupby('LOO sample')
    .agg(
        mean_error = ('abs_error', 'mean'),
        var_true    = ('true', 'var'),
    )
    .reset_index()
)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=sample_stats,
    x='var_true',
    y='mean_error',
    s=25,
    edgecolor='k'
)
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.title('Sample‑wise: Mean Prediction Error vs Variance of True Values')
plt.xlabel('Variance of True Values')
plt.ylabel('Mean Prediction Error')
plt.tight_layout()
plt.show()

## Boxplot Protein and Samples

In [ ]:
# Visually enhanced box/violin plot of MSE: proteins vs samples

df = combined_df.copy()
sample_col = "LOO sample" if "LOO sample" in df.columns else "sample"

# ensure squared_error exists
if "squared_error" not in df.columns:
    df["squared_error"] = (df["true"] - df["predicted"])**2

# per-group MSE
protein_mse = (
    df.groupby("protein", sort=False)["squared_error"]
      .mean()
      .reset_index()
      .rename(columns={"squared_error": "mse"})
)
sample_mse = (
    df.groupby(sample_col, sort=False)["squared_error"]
      .mean()
      .reset_index()
      .rename(columns={"squared_error": "mse", sample_col: "sample"})
)

# tidy for plotting
plot_df = pd.concat(
    [protein_mse.assign(group="Proteins")[["group", "mse"]],
     sample_mse.assign(group="Samples")[["group", "mse"]]],
    ignore_index=True
)

# aesthetics
palette = {"Proteins": "#4682B4",  # steelblue
           "Samples":  "#3CB371"}  # mediumseagreen
sns.set_theme(style="whitegrid", context="talk")

# y-limit trimmed to reduce extreme outlier pull
ymax = np.percentile(plot_df["mse"], 99.5)
ymax = float(np.clip(ymax * 1.05, 0.1, None))  # small buffer

fig, ax = plt.subplots(figsize=(8, 5), dpi=110)  # was (18, 6), 140


# violin (soft fill, no inner lines)
sns.violinplot(
    data=plot_df, x="group", y="mse",
    palette=palette, inner=None, cut=0, width=0.9, alpha=0.6, ax=ax
)

# box overlay (thin, clean)
sns.boxplot(
    data=plot_df, x="group", y="mse",
    width=0.32, showcaps=True, showfliers=False, boxprops={"facecolor": "white", "alpha": 0.9},
    medianprops={"linewidth": 2, "color": "#222"},
    whiskerprops={"linewidth": 1.4},
    capprops={"linewidth": 1.4},
    ax=ax
)

# light jittered points (subsample to keep it crisp)
pts = (
    plot_df.groupby("group", group_keys=False)
           .apply(lambda g: g.sample(n=min(500, len(g)), random_state=0))
)
sns.stripplot(
    data=pts, x="group", y="mse",
    color="k", alpha=0.18, size=2, jitter=0.28, ax=ax, zorder=1
)

# titles, labels, and counts
counts = {"Proteins": len(protein_mse), "Samples": len(sample_mse)}
ax.set_title("MSE distributions: proteins vs. samples", pad=10, weight="bold")
ax.set_xlabel("")
ax.set_ylabel("Mean Squared Error (MSE)")
ax.set_xticklabels([
    f"Proteins (n={counts['Proteins']})",
    f"Samples (n={counts['Samples']})"
])

# median annotations
meds = plot_df.groupby("group")["mse"].median().to_dict()
for i, g in enumerate(["Proteins", "Samples"]):
    m = meds[g]
    ax.text(i, m, f"median {m:.3f}", ha="center", va="bottom",
            fontsize=9, weight="bold",
            bbox=dict(boxstyle="round,pad=0.25", facecolor="white", edgecolor="none", alpha=0.8))

# tidy up
ax.set_ylim(0, ymax)
ax.grid(axis="y", linestyle="--", alpha=0.25)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()


# MAE vs Prot- Encoding Gene correlations

In [ ]:
corrs

# Cluster-Wise analysis

In [ ]:
clusters = pd.read_csv("../data/protein_clusters_k3.csv")

In [ ]:
clusters

Distributions

In [ ]:
# Side-by-side KDEs by cluster: per-protein MSE (left) and cosine similarity (right)
# --- config
palette = {1: "#1f77b4", 2: "#ff7f0e", 3: "#2ca02c"}  # blue, orange, green

# --- helpers
def _cos_sim(a, b):
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    denom = na * nb
    return float(np.dot(a, b) / denom) if denom > 0 else np.nan

def _pad_range(x):
    x = np.asarray(x)
    xmin, xmax = float(np.min(x)), float(np.max(x))
    span = max(xmax - xmin, 1e-6)
    pad = 0.02 * span
    return xmin - pad, xmax + pad

# --- inputs
dfc = combined_df.copy()

# ensure squared_error exists
if "squared_error" not in dfc.columns:
    dfc["squared_error"] = (dfc["true"] - dfc["predicted"])**2

# --- per-protein metrics
prot_mse = (
    dfc.groupby("protein", sort=False)["squared_error"]
       .mean()
       .reset_index()
       .rename(columns={"protein": "Protein", "squared_error": "mse"})
)

prot_cos = (
    dfc.groupby("protein", sort=False)
       .apply(lambda g: _cos_sim(g["true"].to_numpy(), g["predicted"].to_numpy()))
       .reset_index(name="cosine_similarity")
       .rename(columns={"protein": "Protein"})
       .dropna(subset=["cosine_similarity"])
)

# merge with cluster mapping (expects columns: Protein, Cluster)
df_mse = prot_mse.merge(clusters[["Protein", "Cluster"]], on="Protein", how="inner")
df_cos = prot_cos.merge(clusters[["Protein", "Cluster"]], on="Protein", how="inner")

# --- x-ranges
xmin_mse, xmax_mse = _pad_range(df_mse["mse"].values)
xmin_cos, xmax_cos = _pad_range(df_cos["cosine_similarity"].values)
xr_mse = np.linspace(xmin_mse, xmax_mse, 512)
xr_cos = np.linspace(xmin_cos, xmax_cos, 512)

# --- plotting
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5), dpi=140)

# Left: MSE KDEs
for c in sorted(df_mse["Cluster"].dropna().unique()):
    vals = df_mse.loc[df_mse["Cluster"] == c, "mse"].to_numpy()
    if len(vals) >= 2:
        kde = gaussian_kde(vals)  # tweak bandwidth via bw_method=0.3 if desired
        ax1.plot(xr_mse, kde(xr_mse), lw=2, color=palette.get(c, "#333"),
                 label=f"Cluster {c} (n={len(vals)})")
    elif len(vals) == 1:  # fallback for very small groups
        ax1.axvline(vals[0], ymax=0.12, lw=2, color=palette.get(c, "#333"),
                    label=f"Cluster {c} (n=1)")

ax1.set_title("Per-protein Mean Squared Error")
ax1.set_xlabel("MSE")
ax1.set_ylabel("Density")
ax1.grid(alpha=0.25)

# Right: Cosine similarity KDEs
for c in sorted(df_cos["Cluster"].dropna().unique()):
    vals = df_cos.loc[df_cos["Cluster"] == c, "cosine_similarity"].to_numpy()
    if len(vals) >= 2:
        kde = gaussian_kde(vals)
        ax2.plot(xr_cos, kde(xr_cos), lw=2, color=palette.get(c, "#333"))
    elif len(vals) == 1:
        ax2.axvline(vals[0], ymax=0.12, lw=2, color=palette.get(c, "#333"))

ax2.set_title("Per-protein Cosine Similarity")
ax2.set_xlabel("Cosine similarity")
ax2.grid(alpha=0.25)

# Shared legend (from left subplot)
handles, labels = ax1.get_legend_handles_labels()
fig.legend(handles, labels, ncol=3, loc="upper center", frameon=False, bbox_to_anchor=(0.5, 1.01))

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


Scatterplot

In [ ]:
# Side-by-side jittered scatters by cluster: per-protein MSE (left) and cosine similarity (right)

# --- config
palette = {1: "#1f77b4", 2: "#ff7f0e", 3: "#2ca02c"}  # steelblue, orange, green

def _cos_sim(a, b):
    na = np.linalg.norm(a); nb = np.linalg.norm(b)
    denom = na * nb
    return float(np.dot(a, b) / denom) if denom > 0 else np.nan

# --- build per-protein metrics from combined_df and merge clusters
dfc = combined_df.copy()
if "squared_error" not in dfc.columns:
    dfc["squared_error"] = (dfc["true"] - dfc["predicted"])**2

# per-protein MSE
df_mse = (
    dfc.groupby("protein", sort=False)["squared_error"]
       .median()
       .reset_index()
       .rename(columns={"protein": "Protein", "squared_error": "mse"})
    .merge(clusters[["Protein", "Cluster"]], on="Protein", how="inner")
)

# per-protein cosine similarity
df_cos = (
    dfc.groupby("protein", sort=False)
       .apply(lambda g: _cos_sim(g["true"].to_numpy(), g["predicted"].to_numpy()))
       .reset_index(name="cosine_similarity")
       .rename(columns={"protein": "Protein"})
       .dropna(subset=["cosine_similarity"])
    .merge(clusters[["Protein", "Cluster"]], on="Protein", how="inner")
)

# --- helper for jittered scatter with median lines
#“Jittered” just means we add a tiny bit of random noise to each point’s position along the x-axis
# so overlapping points don’t sit exactly on top of each other. 
# It makes dense, discrete groups (like clusters 1/2/3) easier to read.
def jitter_scatter(ax, clusters_col, y_col, title, ylabel, seed=0):
    rng = np.random.default_rng(seed)
    x = clusters_col.astype(int).to_numpy()
    y = y_col.to_numpy()

    # horizontal jitter so points don't overlap
    x_jittered = x + rng.normal(0, 0.06, size=len(x))
    colors = [palette.get(int(c), "#333") for c in x]

    ax.scatter(x_jittered, y, s=18, alpha=0.65, c=colors)

    # per-cluster medians
    for c in sorted(pd.unique(clusters_col)):
        yc = y_col[clusters_col == c]
        if len(yc) == 0: 
            continue
        med = float(np.median(yc))
        ax.hlines(med, c - 0.27, c + 0.27, colors=palette.get(int(c), "#333"), linewidth=3)
        ax.text(c + 0.3, med, f"median={med:.3f}", fontsize=9, va="center")

    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(["Cluster 1", "Cluster 2", "Cluster 3"])
    ax.set_xlim(0.5, 3.5)
    ax.set_title(title)
    ax.set_xlabel("")
    ax.set_ylabel(ylabel)
    ax.grid(axis="y", alpha=0.3)

# --- plot side-by-side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5.2), dpi=140)

jitter_scatter(ax1, df_mse["Cluster"], df_mse["mse"],
               title="Per-protein MedianSE by Cluster",
               ylabel="Median squared error (per protein)",
               seed=0)

jitter_scatter(ax2, df_cos["Cluster"], df_cos["cosine_similarity"],
               title="Per-protein Cosine Similarity by Cluster",
               ylabel="Cosine similarity (per protein)",
               seed=1)  # different seed to vary jitter pattern

plt.tight_layout()
plt.show()
